In [2]:
import pandas as pd

In [6]:
pd.set_option('display.max_columns', None)

In [4]:
week1 = pd.read_csv('../mainexperiment/weekly/tier_merged_updated/tiers_merged_all-0710-0713.csv')

/var/folders/s_/0td42yfs6wjd9wdpc5nr90400000gn/T/ipykernel_51472/3385163340.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  week1 = pd.read_csv('../mainexperiment/weekly/tier_merged_updated/tiers_merged_all-0710-0713.csv')


In [5]:
week1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15566 entries, 0 to 15565
Data columns (total 88 columns):
 #   Column                                                                                 Non-Null Count  Dtype  
---  ------                                                                                 --------------  -----  
 0   Unnamed: 0.1                                                                           15566 non-null  int64  
 1   Unnamed: 0                                                                             15566 non-null  int64  
 2   date                                                                                   15566 non-null  object 
 3   region                                                                                 15566 non-null  object 
 4   zone                                                                                   15566 non-null  object 
 5   driver_id                                                                 

In [5]:
week1.head()

,Unnamed: 0.1,Unnamed: 0,date,region,zone,driver_id,rider_level,most_freq_district,phone,driver_tier,...,extra_non_wage_cost,peak_extra_non_wage_cost,offpeak_extra_non_wage_cost,region::multi-filter,avg_rain,Rain_Level,Rider Group,Start Date,End Date,Incentive
0,0,0,2025-07-11,CHANTHABURI,N,LMD07LE7W,7,Mueang Chanthaburi,923394134.0,MASTER,...,0.0,NaN,0.0,CHANTHABURI,0.00,no-rain,Top,2025-07-10,2025-07-13,No Incentive
1,1,1,2025-07-12,CHANTHABURI,N,LMD07LE7W,7,Mueang Chanthaburi,923394134.0,MASTER,...,0.0,0.0,0.0,CHANTHABURI,0.00,no-rain,Top,2025-07-10,2025-07-13,No Incentive
2,2,2,2025-07-10,CHANTHABURI,N,LMD07LE7W,8,Mueang Chanthaburi,923394134.0,MASTER,...,0.0,0.0,0.0,CHANTHABURI,0.37,Light,Top,2025-07-10,2025-07-13,No Incentive
3,3,3,2025-07-11,CHANTHABURI,N,LMD0F7D6F,6,Mueang Chanthaburi,634519461.0,GOLD,...,0.0,0.0,0.0,CHANTHABURI,0.00,no-rain,Mid,2025-07-10,2025-07-13,No Incentive
4,4,4,2025-07-13,CHANTHABURI,N,LMD0N5L6G,0,Mueang Chanthaburi,946537473.0,SILVER,...,0.0,0.0,0.0,CHANTHABURI,0.00,no-rain,Bottom,2025-07-10,2025-07-13,No Incentive


In [8]:
!pip install statsmodels pandas numpy


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [11]:
"""
Difference-in-Differences Panel Regression
==========================================================================
Design:
  - W1 (Jul 10-13) = clean no-incentive baseline for ALL drivers
  - W4 (Jul 28-Aug 10) = second no-incentive reset week for ALL drivers
  - W2, W3, W5, W6, W7 = drivers receive varying incentives week by week
  - Same drivers observed across all weeks

Model:
  completed_per_day_it = α_i + δ_t + β·Incentive_it + ε_it

  Where:
    α_i  = driver fixed effects  (absorbs each driver's baseline productivity)
    δ_t  = week fixed effects    (absorbs week-level demand shocks, weather, etc.)
    β    = causal effect of each incentive vs no-incentive weeks
    SE   = clustered by zone

  This is a two-way fixed effects (TWFE) DiD:
  "How many MORE orders per day did a driver complete in weeks they had
   incentive X compared to their own no-incentive baseline weeks?"

  OUTCOME: completed_per_day (orders / active days in that week)
  This normalises for W4 being 14 days vs ~7 days for other weeks.

Models:
  1. TWFE pooled          — main causal estimate (driver FE + week FE)
  2. Pooled OLS           — week FE only, no driver FE (comparison)
  3. TWFE per week        — heterogeneous effects over time
  4. TWFE × zone_type     — heterogeneous effects by win/lose/swing region

Outputs:
  did_incentive_results.csv      — tidy results (coef, SE, pval, CI per incentive)
  did_incentive_pivot.csv        — coef + pval side by side
  did_regression_summary.txt     — formatted regression table
  did_zone_heterogeneity.csv     — Model 4 interaction results by zone type
"""

import os
import re
import glob
import warnings
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

warnings.filterwarnings("ignore")

# ═══════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════

DATA_DIR   = "../mainexperiment/weekly/tier_merged_updated/"
OUTPUT_DIR = "./results"

# Use per-day outcome to normalise across weeks of different lengths
# The raw 'completed' column will be divided by active days per driver-week
OUTCOME_RAW   = "completed"           # raw column in data
OUTCOME       = "completed_per_day"   # derived outcome used in models

TREATMENT_COL = "Incentive"
DRIVER_ID_COL = "driver_id"
DAYS_COL      = "active_days"         # column with days active that week (if available)
                                       # if missing, will be inferred from row counts

# Zone type column — must contain values like 'Win', 'Lose', 'Swing'
# (exact strings — update to match your data)
ZONE_TYPE_COL = "status"

# The exact string in Incentive column meaning "no incentive"
REFERENCE_LEVEL = "No Incentive"

CLUSTER_BY = "status"

# Week labels from filenames — W1 must come first (baseline)
WEEK_MAP = [
    ("0710_0713", "W1_Jul10"),    # ← baseline: no incentive for everyone
    ("0714_0720", "W2_Jul14"),
    ("0721_0727", "W3_Jul21"),
    ("0728_0810", "W4_Jul28"),    # ← second reset: no incentive for everyone (14 days)
    ("0811_0815", "W5_Aug11"),
    ("0816_0822", "W6_Aug16"),
    ("0823_0829", "W7_Aug23"),
]

# Expected days per week (used to infer active_days if column is missing)
WEEK_DAYS = {
    "W1_Jul10": 4,
    "W2_Jul14": 7,
    "W3_Jul21": 7,
    "W4_Jul28": 14,   # two-week reset period
    "W5_Aug11": 5,
    "W6_Aug16": 7,
    "W7_Aug23": 7,
}

# ═══════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════

def safe(c):
    """Make column names safe for statsmodels formulas."""
    return re.sub(r"[^0-9a-zA-Z_]", "_", str(c))

def sig(p):
    if p < 0.01:  return "***"
    if p < 0.05:  return "**"
    if p < 0.10:  return "*"
    return ""

def load_files(data_dir):
    pattern = os.path.join(data_dir, "tiers_merged_all-*.csv")
    files   = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(
            f"No files matching 'tiers_merged_all-*.csv' found in: {os.path.abspath(data_dir)}"
        )
    dfs = []
    for fp in files:
        fname      = os.path.basename(fp)
        fname_norm = fname.replace("-", "_")
        week_label = next((lbl for pat, lbl in WEEK_MAP if pat in fname_norm), fname)
        df         = pd.read_csv(fp, low_memory=False)
        df["_week"] = week_label
        dfs.append(df)
        print(f"  {fname:<45} → {week_label}  ({len(df):,} rows)")
    out = pd.concat(dfs, ignore_index=True)
    print(f"\n  Total rows: {len(out):,}")
    return out

def aggregate_driver_week(df, outcome_raw_s, treatment_s, driver_s,
                          cluster_s, week_s, days_col_s, zone_type_s):
    """
    Collapse to one row per driver × week.
    - completed:       summed across days
    - active_days:     summed (or inferred from row count if column missing)
    - Incentive/zone:  mode (most common value that week)
    - zone_type:       mode
    """
    cat_cols = [c for c in [treatment_s, cluster_s, zone_type_s] if c and c in df.columns]
    has_days = days_col_s and days_col_s in df.columns

    agg_dict = {outcome_raw_s: "sum"}
    if has_days:
        agg_dict[days_col_s] = "sum"
    else:
        # proxy: count rows (each row = 1 driver-day)
        agg_dict["_row_count"] = ("_row_count", "sum") if "_row_count" in df.columns else None

    for c in cat_cols:
        agg_dict[c] = lambda x: x.mode().iloc[0] if len(x.mode()) else np.nan

    # Add a row count column to use as day proxy if needed
    df = df.copy()
    df["_row_count"] = 1

    if not has_days:
        agg_dict["_row_count"] = "sum"

    agg = df.groupby([driver_s, week_s], as_index=False).agg(agg_dict)

    # Build completed_per_day
    if has_days:
        # Use actual active days
        agg[OUTCOME] = agg[outcome_raw_s] / agg[days_col_s].replace(0, np.nan)
    else:
        # Use week-level expected days as denominator (more stable than row count)
        agg["_expected_days"] = agg[week_s].map(WEEK_DAYS).fillna(7)
        agg[OUTCOME] = agg[outcome_raw_s] / agg["_expected_days"]

    print(f"  Rows after aggregation: {len(agg):,}")
    print(f"  Unique drivers:         {agg[driver_s].nunique():,}")
    print(f"  Unique weeks:           {agg[week_s].nunique()}")
    return agg

def run_ols_clustered(formula, data, cluster_col):
    if cluster_col and cluster_col in data.columns and data[cluster_col].nunique() > 1:
        return smf.ols(formula, data=data).fit(
            cov_type="cluster",
            cov_kwds={"groups": data[cluster_col]},
        )
    return smf.ols(formula, data=data).fit(cov_type="HC3")

def demean_within_driver(panel, outcome_s, driver_s):
    """Within-transformation: demean by driver (equivalent to driver FE dummies)."""
    demeaned = (outcome_s + "_dm")
    panel[demeaned] = (
        panel[outcome_s]
        - panel.groupby(driver_s)[outcome_s].transform("mean")
        + panel[outcome_s].mean()
    )
    return panel, demeaned

def extract_incentive_coefs(model, treatment_s, ref):
    rows = []
    for param in model.params.index:
        if treatment_s in param and "Intercept" not in param:
            match = re.search(r"\[T\.(.*?)\]", param)
            label = match.group(1) if match else param
            ci    = model.conf_int().loc[param]
            rows.append({
                "incentive": label,
                "coef":    round(model.params[param],  4),
                "se":      round(model.bse[param],     4),
                "tstat":   round(model.tvalues[param], 4),
                "pval":    round(model.pvalues[param], 4),
                "ci_low":  round(ci[0], 4),
                "ci_high": round(ci[1], 4),
            })
    return rows

def print_coef_table(rows, ref):
    print(f"\n  {'Incentive':<38} {'Coef':>9} {'SE':>7} {'p':>7}  Sig  Interpretation")
    print("  " + "─" * 85)
    for r in sorted(rows, key=lambda x: -x["coef"]):
        direction = (f"+{r['coef']:.2f} orders/day vs {ref}"
                     if r["coef"] >= 0
                     else f"{r['coef']:.2f} orders/day vs {ref}")
        print(f"  {r['incentive']:<38} {r['coef']:>9.4f} {r['se']:>7.4f}"
              f" {r['pval']:>7.3f}  {sig(r['pval']):<3}  {direction}")

# ═══════════════════════════════════════════════════════
# MODEL 4 HELPERS
# ═══════════════════════════════════════════════════════

def extract_interaction_coefs(model, treatment_s, zone_type_s, ref):
    """
    Extract coefficients from the incentive × zone_type interaction model.

    Statsmodels encodes interactions like:
      C(Incentive)[T.Active Bonus]:C(zone_type)[T.Swing]

    This function parses those into tidy rows with columns:
      incentive, zone_type, coef, se, tstat, pval, ci_low, ci_high

    Main effects (incentive without zone interaction) are also extracted
    and labelled with zone_type = '<reference zone>' for clarity.
    """
    rows = []
    ci_df = model.conf_int()

    for param in model.params.index:
        # Skip intercept and pure week FE terms
        if "Intercept" in param or (week_s_global in param and treatment_s not in param
                                     and zone_type_s not in param):
            continue

        has_incentive  = treatment_s  in param
        has_zone       = zone_type_s  in param

        if not has_incentive:
            continue

        # Parse incentive label
        inc_match = re.search(rf"{treatment_s}\[T\.(.*?)\]", param)
        incentive_label = inc_match.group(1) if inc_match else param

        # Parse zone_type label (present only in interaction terms)
        zone_match = re.search(rf"{zone_type_s}\[T\.(.*?)\]", param)
        zone_label = zone_match.group(1) if zone_match else "reference_zone"

        ci = ci_df.loc[param]
        rows.append({
            "incentive":  incentive_label,
            "zone_type":  zone_label,
            "term":       "interaction" if has_zone else "main_effect",
            "coef":       round(model.params[param],  4),
            "se":         round(model.bse[param],     4),
            "tstat":      round(model.tvalues[param], 4),
            "pval":       round(model.pvalues[param], 4),
            "ci_low":     round(ci[0], 4),
            "ci_high":    round(ci[1], 4),
        })
    return rows

def compute_zone_marginal_effects(interaction_rows, zone_types, ref_zone):
    """
    Convert main effects + interactions into total marginal effects per zone.

    For the reference zone: effect = main_effect coef
    For other zones:        effect = main_effect coef + interaction coef

    This gives you the answer to "what is the effect of Active Bonus
    specifically in Win / Lose / Swing zones?"
    """
    df = pd.DataFrame(interaction_rows)
    main   = df[df["term"] == "main_effect"].set_index("incentive")
    inter  = df[df["term"] == "interaction"].set_index(["incentive", "zone_type"])

    results = []
    incentives = main.index.unique()

    for inc in incentives:
        # Reference zone — main effect only
        m = main.loc[inc]
        results.append({
            "incentive":     inc,
            "zone_type":     ref_zone,
            "total_coef":    m["coef"],
            "pval":          m["pval"],
            "note":          "main effect (reference zone)"
        })

        # Other zones — main effect + interaction
        for zone in zone_types:
            if zone == ref_zone:
                continue
            if (inc, zone) in inter.index:
                i = inter.loc[(inc, zone)]
                total = m["coef"] + i["coef"]
                results.append({
                    "incentive":     inc,
                    "zone_type":     zone,
                    "total_coef":    round(total, 4),
                    "interaction_coef": i["coef"],
                    "interaction_pval": i["pval"],
                    "pval":          i["pval"],   # significance of the *difference*
                    "note":          "main + interaction"
                })
            else:
                results.append({
                    "incentive":  inc,
                    "zone_type":  zone,
                    "total_coef": m["coef"],
                    "pval":       np.nan,
                    "note":       "no interaction term found"
                })

    return pd.DataFrame(results)

def print_zone_effects_table(marginal_df, ref_zone):
    print(f"\n  Total effect of each incentive by zone type")
    print(f"  (reference zone = '{ref_zone}'; significance = interaction p-value for non-ref zones)")
    print(f"\n  {'Incentive':<28} {'Zone':<10} {'Effect':>8}  Sig  Note")
    print("  " + "─" * 70)
    for _, r in marginal_df.sort_values(["incentive", "zone_type"]).iterrows():
        p = r.get("pval", np.nan)
        s = sig(p) if pd.notna(p) else ""
        print(f"  {r['incentive']:<28} {r['zone_type']:<10} {r['total_coef']:>8.3f}  {s:<3}  {r['note']}")

# ═══════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════

# Global reference for week column name (used in interaction extractor)
week_s_global = "_week"

def main():
    global week_s_global

    print("=" * 70)
    print("Thailand Incentive Experiment — DiD Panel Regression")
    print("Two-Way Fixed Effects: Driver FE + Week FE")
    print("Outcome: completed orders per active day")
    print("=" * 70 + "\n")

    # ── Load ──────────────────────────────────────────────
    df = load_files(DATA_DIR)

    # ── Safe column names ─────────────────────────────────
    needed = [OUTCOME_RAW, TREATMENT_COL, DRIVER_ID_COL, CLUSTER_BY,
              DAYS_COL, ZONE_TYPE_COL, "_week"]
    rename = {c: safe(c) for c in needed if c and c in df.columns}
    df = df.rename(columns=rename)

    outcome_raw_s = rename.get(OUTCOME_RAW,   safe(OUTCOME_RAW))
    outcome_s     = safe(OUTCOME)
    treatment_s   = rename.get(TREATMENT_COL, safe(TREATMENT_COL))
    driver_s      = rename.get(DRIVER_ID_COL, safe(DRIVER_ID_COL))
    cluster_s     = rename.get(CLUSTER_BY,    safe(CLUSTER_BY)) if CLUSTER_BY else None
    days_col_s    = rename.get(DAYS_COL,      safe(DAYS_COL))   if DAYS_COL else None
    zone_type_s   = rename.get(ZONE_TYPE_COL, safe(ZONE_TYPE_COL)) if ZONE_TYPE_COL else None
    week_s        = "_week"
    week_s_global = week_s

    df[outcome_raw_s] = pd.to_numeric(df[outcome_raw_s], errors="coerce")

    # ── Incentive distribution ────────────────────────────
    print("\n" + "─" * 70)
    print("Incentive × Week distribution (driver-day rows):")
    cross = pd.crosstab(df[week_s], df[treatment_s])
    print(cross.to_string())

    # ── Zone type distribution ────────────────────────────
    if zone_type_s and zone_type_s in df.columns:
        print("\nZone type distribution:")
        print(df[zone_type_s].value_counts().to_string())
    else:
        print(f"\n  ⚠ Column '{ZONE_TYPE_COL}' not found — Model 4 will be skipped.")
        zone_type_s = None

    # ── Resolve reference ─────────────────────────────────
    ref = REFERENCE_LEVEL
    if ref not in df[treatment_s].values:
        w1 = next((lbl for _, lbl in WEEK_MAP if lbl in df[week_s].unique()), None)
        ref = df[df[week_s] == w1][treatment_s].mode().iloc[0] if w1 else \
              df[treatment_s].value_counts().idxmax()
        print(f"\n  '{REFERENCE_LEVEL}' not found — using '{ref}' as reference")
    print(f"\n  ✓ Reference (baseline): '{ref}'")

    # ── Aggregate to driver × week ────────────────────────
    print("\n" + "─" * 70)
    print("Aggregating to driver × week panel...")
    panel = aggregate_driver_week(df, outcome_raw_s, treatment_s, driver_s,
                                  cluster_s, week_s, days_col_s, zone_type_s)
    panel = panel.dropna(subset=[outcome_s, treatment_s])

    # ── Sanity checks ─────────────────────────────────────
    print("\n" + "─" * 70)
    print("Panel sanity checks:")
    obs_per_driver = panel.groupby(driver_s).size()
    print(f"  Weeks observed per driver — mean: {obs_per_driver.mean():.1f} "
          f"min: {obs_per_driver.min()}  max: {obs_per_driver.max()}")

    print("\n  Incentive × Week counts (driver-week rows):")
    cross2 = pd.crosstab(panel[week_s], panel[treatment_s])
    print(cross2.to_string())

    print("\n  Mean completed orders/day by incentive (raw, unadjusted):")
    means = panel.groupby(treatment_s)[outcome_s].agg(["mean", "count"]).round(3)
    means.columns = ["mean_completed_per_day", "n_driver_weeks"]
    print(means.to_string())

    if zone_type_s and zone_type_s in panel.columns:
        print("\n  No-incentive driver zone composition by week:")
        no_inc = panel[panel[treatment_s] == ref]
        zone_cross = pd.crosstab(no_inc[week_s], no_inc[zone_type_s])
        print(zone_cross.to_string())
        print("  ↑ Check if W5/W6 no-incentive drivers are concentrated in Win zones")

    # ── Driver FE demeaning ───────────────────────────────
    panel, demeaned_col = demean_within_driver(panel, outcome_s, driver_s)

    treatment_expr = f"C({treatment_s}, Treatment('{ref}'))"

    # ══════════════════════════════════════════════════════
    # MODEL 1 — TWFE DiD (driver FE + week FE)
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("MODEL 1: Two-Way Fixed Effects DiD")
    print(f"  {OUTCOME} ~ C(Incentive) + C(driver_id) + C(week)")
    print("=" * 70)

    formula_twfe  = f"{demeaned_col} ~ {treatment_expr} + C({week_s})"
    twfe_model    = run_ols_clustered(formula_twfe, panel, cluster_s)
    twfe_rows     = extract_incentive_coefs(twfe_model, treatment_s, ref)

    print(f"\n  n={int(twfe_model.nobs):,}  drivers={panel[driver_s].nunique():,}  "
          f"R²={twfe_model.rsquared:.3f}  Adj-R²={twfe_model.rsquared_adj:.3f}")
    print_coef_table(twfe_rows, ref)

    # ══════════════════════════════════════════════════════
    # MODEL 2 — Pooled OLS (week FE only)
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("MODEL 2: Pooled OLS (week FE only, no driver FE) — for comparison")
    print("=" * 70)

    formula_pooled = f"{outcome_s} ~ {treatment_expr} + C({week_s})"
    pooled_model   = run_ols_clustered(formula_pooled, panel, cluster_s)
    pooled_rows    = extract_incentive_coefs(pooled_model, treatment_s, ref)

    print(f"\n  n={int(pooled_model.nobs):,}  "
          f"R²={pooled_model.rsquared:.3f}  Adj-R²={pooled_model.rsquared_adj:.3f}")
    print_coef_table(pooled_rows, ref)

    print("\n  ↑ If MODEL 1 vs MODEL 2 coefficients shift a lot,")
    print("    driver FE is doing important work (selection bias exists).")

    # ══════════════════════════════════════════════════════
    # MODEL 3 — TWFE per week (heterogeneous effects over time)
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("MODEL 3: TWFE per week (heterogeneous treatment effects over time)")
    print("=" * 70)

    week_order    = [lbl for _, lbl in WEEK_MAP if lbl in panel[week_s].unique()]
    week_models   = {}
    all_week_rows = []

    for week in week_order:
        sub = panel[
            (panel[week_s] == week) |
            (panel[treatment_s] == ref)
        ].copy()

        sub, sub_dm = demean_within_driver(sub, outcome_s, driver_s)

        n_incentives = sub[sub[week_s] == week][treatment_s].nunique()
        if n_incentives < 2:
            only = sub[sub[week_s] == week][treatment_s].iloc[0]
            label = "skipped (all on No Incentive)" if only == ref else \
                    f"skipped (only incentive: '{only}')"
            print(f"\n  {week}: {label}")
            continue

        try:
            formula_w = f"{sub_dm} ~ {treatment_expr} + C({week_s})"
            model_w   = run_ols_clustered(formula_w, sub, cluster_s)
            week_models[week] = model_w

            rows = extract_incentive_coefs(model_w, treatment_s, ref)
            week_incentives = sub[sub[week_s] == week][treatment_s].unique()
            rows = [r for r in rows if r["incentive"] in week_incentives]

            for r in rows:
                r.update({"week": week, "n_obs": int(model_w.nobs),
                           "r2": round(model_w.rsquared, 4), "reference": ref,
                           "model": "TWFE_by_week"})
            all_week_rows.extend(rows)

            print(f"\n  {week}  (n={int(model_w.nobs):,}  R²={model_w.rsquared:.3f})")
            print_coef_table(rows, ref)

        except Exception as e:
            print(f"\n  {week}: ERROR — {e}")

    # ══════════════════════════════════════════════════════
    # MODEL 4 — TWFE × zone_type (heterogeneous effects by region)
    #
    # Formula:
    #   completed_per_day_dm ~ C(Incentive) * C(zone_type) + C(week)
    #
    # This adds an interaction between incentive and zone type on top of
    # the driver-demeaned outcome (driver FE already absorbed via demeaning).
    #
    # Interpretation:
    #   Main effect of incentive    = effect in the REFERENCE zone
    #   Interaction term            = HOW MUCH MORE/LESS the effect is in
    #                                 non-reference zones vs the reference zone
    #   Total effect in a zone      = main effect + interaction
    #                                 (computed in compute_zone_marginal_effects)
    #
    # This answers: "Does Active Bonus work better in Swing zones than Win zones?"
    # ══════════════════════════════════════════════════════
    zone_rows_all   = []
    marginal_df_out = None

    if zone_type_s and zone_type_s in panel.columns:
        print("\n" + "=" * 70)
        print("MODEL 4: TWFE × Zone Type (heterogeneous effects by Win/Lose/Swing)")
        print(f"  {OUTCOME}_dm ~ C(Incentive) * C(zone_type) + C(week)")
        print("  Main effect   = incentive effect in the REFERENCE zone")
        print("  Interaction   = differential effect in other zones vs reference")
        print("  Total effect  = main + interaction (reported in marginal table)")
        print("=" * 70)

        zone_types = sorted(panel[zone_type_s].dropna().unique())

        # Choose reference zone — use the largest group for stability
        ref_zone = panel[zone_type_s].value_counts().idxmax()
        zone_type_expr = f"C({zone_type_s}, Treatment('{ref_zone}'))"

        print(f"\n  Zone types found: {zone_types}")
        print(f"  Reference zone (largest): '{ref_zone}'")
        print(f"\n  Zone × Week distribution:")
        if zone_type_s in panel.columns:
            print(pd.crosstab(panel[week_s], panel[zone_type_s]).to_string())

        formula_zone = (
            f"{demeaned_col} ~ "
            f"{treatment_expr} * {zone_type_expr} + "
            f"C({week_s})"
        )

        try:
            zone_model = run_ols_clustered(formula_zone, panel, cluster_s)

            print(f"\n  n={int(zone_model.nobs):,}  "
                  f"R²={zone_model.rsquared:.3f}  "
                  f"Adj-R²={zone_model.rsquared_adj:.3f}")

            # ── Raw interaction coefficients ───────────────
            zone_rows = extract_interaction_coefs(zone_model, treatment_s,
                                                   zone_type_s, ref)
            zone_rows_all = zone_rows

            print(f"\n  Raw coefficients (main effects + interactions):")
            print(f"  {'Term':<60} {'Coef':>9} {'SE':>7} {'p':>7}  Sig")
            print("  " + "─" * 90)
            for r in zone_rows:
                term_label = f"{r['incentive']} × {r['zone_type']}" \
                             if r["term"] == "interaction" \
                             else f"{r['incentive']} [main / ref zone]"
                print(f"  {term_label:<60} {r['coef']:>9.4f} {r['se']:>7.4f}"
                      f" {r['pval']:>7.3f}  {sig(r['pval']):<3}")

            # ── Marginal total effects per zone ───────────
            print(f"\n  ── Marginal total effects (main + interaction) ──")
            marginal_df = compute_zone_marginal_effects(zone_rows, zone_types, ref_zone)
            marginal_df_out = marginal_df
            print_zone_effects_table(marginal_df, ref_zone)

            # ── Pivot: incentive × zone for easy scanning ──
            print(f"\n  ── Pivot: total effect by incentive × zone ──")
            pivot = marginal_df.pivot_table(
                index="incentive", columns="zone_type", values="total_coef"
            )
            print(pivot.round(3).to_string())

            print(f"\n  ── Interpretation guide ──")
            print(f"  • Positive value = incentive RAISES orders/day vs No Incentive")
            print(f"  • Interaction p-value = is the zone difference statistically significant?")
            print(f"  • Large variation across columns = zone type matters a lot")
            print(f"  • Small variation across columns = incentive effect is uniform")

        except Exception as e:
            print(f"\n  Model 4 ERROR — {e}")
            import traceback
            traceback.print_exc()

    else:
        print("\n  MODEL 4 skipped — zone_type column not available.")
        print(f"  Add '{ZONE_TYPE_COL}' column to your data to enable this analysis.")

    # ══════════════════════════════════════════════════════
    # SAVE OUTPUTS
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("Saving outputs...")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Tag model rows
    for r in twfe_rows:
        r.update({"week": "TWFE_POOLED", "n_obs": int(twfe_model.nobs),
                   "r2": round(twfe_model.rsquared, 4), "reference": ref,
                   "model": "TWFE"})
    for r in pooled_rows:
        r.update({"week": "OLS_POOLED", "n_obs": int(pooled_model.nobs),
                   "r2": round(pooled_model.rsquared, 4), "reference": ref,
                   "model": "OLS_no_driverFE"})

    results_df = pd.DataFrame(twfe_rows + pooled_rows + all_week_rows)
    results_df = results_df[[
        "model", "week", "incentive", "coef", "se", "tstat",
        "pval", "ci_low", "ci_high", "n_obs", "r2", "reference"
    ]].round(4)

    csv1 = os.path.join(OUTPUT_DIR, "did_incentive_results_orders_completed.csv")
    results_df.to_csv(csv1, index=False)
    print(f"\n  ✓ {csv1}")

    # Pivot: TWFE weekly coefs × incentive
    week_rows_df = pd.DataFrame(all_week_rows)
    if not week_rows_df.empty:
        pivot_coef = week_rows_df.pivot_table(index="incentive", columns="week", values="coef")
        pivot_pval = week_rows_df.pivot_table(index="incentive", columns="week", values="pval")

        actual_weeks = list(pivot_coef.columns)
        ordered_cols = [w for w in week_order if w in actual_weeks]
        pivot_coef = pivot_coef[ordered_cols]
        pivot_pval = pivot_pval[ordered_cols]

        twfe_series = pd.Series({r["incentive"]: r["coef"] for r in twfe_rows})
        pval_series = pd.Series({r["incentive"]: r["pval"] for r in twfe_rows})
        pivot_coef.insert(0, "TWFE_POOLED", twfe_series)
        pivot_pval.insert(0, "TWFE_POOLED", pval_series)

        csv2 = os.path.join(OUTPUT_DIR, "did_incentive_pivot_orders_completed.csv")
        with open(csv2, "w") as f:
            f.write(f"COEFFICIENTS (orders/day) — vs '{ref}' (driver FE + week FE)\n")
            pivot_coef.round(4).to_csv(f)
            f.write(f"\nP-VALUES\n")
            pivot_pval.round(4).to_csv(f)
        print(f"  ✓ {csv2}")

    # Zone heterogeneity CSV
    if zone_rows_all:
        csv3 = os.path.join(OUTPUT_DIR, "did_zone_heterogeneity_orders_completed.csv")
        zone_raw_df = pd.DataFrame(zone_rows_all)
        zone_raw_df["model"]     = "TWFE_zone_interaction"
        zone_raw_df["reference"] = ref
        zone_raw_df.round(4).to_csv(csv3, index=False)
        print(f"  ✓ {csv3}  (raw interaction coefficients)")

    if marginal_df_out is not None:
        csv4 = os.path.join(OUTPUT_DIR, "did_zone_marginal_effects_orders_compeleted.csv")
        marginal_df_out.round(4).to_csv(csv4, index=False)
        print(f"  ✓ {csv4}  (total effects per incentive × zone)")

    # Regression summary txt
    try:
        all_models      = [twfe_model, pooled_model] + list(week_models.values())
        all_model_names = ["TWFE", "OLS"] + list(week_models.keys())
        incent_params   = [p for p in twfe_model.params.index if treatment_s in p]

        summary = summary_col(
            all_models,
            model_names=all_model_names,
            stars=True,
            float_format="%.3f",
            regressor_order=incent_params,
            info_dict={
                "N":  lambda m: f"{int(m.nobs):,}",
                "R²": lambda m: f"{m.rsquared:.3f}",
            },
        )
        txt = os.path.join(OUTPUT_DIR, "did_regression_summary.txt")
        with open(txt, "w") as f:
            f.write("Two-Way Fixed Effects DiD — Thailand Incentive Experiment\n")
            f.write(f"Outcome: completed orders per active day\n")
            f.write(f"Reference: '{ref}' | SE clustered by {CLUSTER_BY}\n")
            f.write("TWFE = driver FE (within-demeaned) + week FE\n")
            f.write("OLS  = week FE only (no driver FE) — comparison\n\n")
            f.write(str(summary))
        print(f"  ✓ {txt}")
    except Exception as e:
        print(f"  (summary table skipped: {e})")

    print("\n" + "=" * 70)
    print("Done ✓")
    print("─" * 70)
    print("Outputs:")
    print("  did_incentive_results.csv      — full tidy results (Models 1–3)")
    print("  did_incentive_pivot.csv        — incentive × week coefficient matrix")
    print("  did_regression_summary.txt     — paper-style table")
    print("  did_zone_heterogeneity.csv     — Model 4 raw interaction coefficients")
    print("  did_zone_marginal_effects.csv  — Model 4 total effects per zone")
    print("─" * 70)
    print("How to read results:")
    print(f"  coef = extra completed orders/day vs '{ref}',")
    print("         controlling for each driver's own baseline (TWFE model)")
    print("  Model 4: interaction p-value = is the zone difference significant?")
    print("  *** p<0.01  ** p<0.05  * p<0.10")
    print("=" * 70)


if __name__ == "__main__":
    main()

Thailand Incentive Experiment — DiD Panel Regression
Two-Way Fixed Effects: Driver FE + Week FE
Outcome: completed orders per active day

  tiers_merged_all-0710-0713.csv                → W1_Jul10  (15,566 rows)
  tiers_merged_all-0714-0720.csv                → W2_Jul14  (29,301 rows)
  tiers_merged_all-0721-0727.csv                → W3_Jul21  (29,583 rows)
  tiers_merged_all-0728-0810.csv                → W4_Jul28  (63,437 rows)
  tiers_merged_all-0811-0815.csv                → W5_Aug11  (20,559 rows)
  tiers_merged_all-0816-0822.csv                → W6_Aug16  (30,336 rows)
  tiers_merged_all-0823-0829.csv                → W7_Aug23  (30,149 rows)

  Total rows: 218,931

──────────────────────────────────────────────────────────────────────
Incentive × Week distribution (driver-day rows):
Incentive  Active Bonus  Daily Productivity  Guarantee  No Incentive  Streak
_week                                                                       
W1_Jul10              0                   0   

In [12]:
pd.read_csv('./results/did_incentive_results_orders_completed.csv')

,model,week,incentive,coef,se,tstat,pval,ci_low,ci_high,n_obs,r2,reference
0,TWFE,TWFE_POOLED,Active Bonus,0.4942,0.3098,1.5954,0.1106,-0.1129,1.1013,41231,0.0149,No Incentive
1,TWFE,TWFE_POOLED,Daily Productivity,0.3534,0.0393,8.9895,0.0000,0.2763,0.4304,41231,0.0149,No Incentive
2,TWFE,TWFE_POOLED,Guarantee,-0.6893,0.0504,-13.6855,0.0000,-0.7880,-0.5906,41231,0.0149,No Incentive
3,TWFE,TWFE_POOLED,Streak,0.3651,0.0822,4.4414,0.0000,0.2040,0.5262,41231,0.0149,No Incentive
4,OLS_no_driverFE,OLS_POOLED,Active Bonus,0.8076,0.6399,1.2620,0.2069,-0.4466,2.0618,41231,0.0087,No Incentive
5,OLS_no_driverFE,OLS_POOLED,Daily Productivity,0.8243,0.5594,1.4735,0.1406,-0.2722,1.9208,41231,0.0087,No Incentive
6,OLS_no_driverFE,OLS_POOLED,Guarantee,-0.4672,0.3732,-1.2518,0.2106,-1.1988,0.2643,41231,0.0087,No Incentive
7,OLS_no_driverFE,OLS_POOLED,Streak,0.8054,0.1827,4.4079,0.0000,0.4473,1.1635,41231,0.0087,No Incentive
8,TWFE_by_week,W2_Jul14,Active Bonus,-0.1177,0.1227,-0.9590,0.3376,-0.3581,0.1228,22830,0.0205,No Incentive
9,TWFE_by_week,W2_Jul14,Daily Productivity,0.5726,0.2913,1.9661,0.0493,0.0018,1.1435,22830,0.0205,No Incentive


In [13]:
"""
Difference-in-Differences Panel Regression
==========================================================================
Design:
  - W1 (Jul 10-13) = clean no-incentive baseline for ALL drivers
  - W4 (Jul 28-Aug 10) = second no-incentive reset week for ALL drivers
  - W2, W3, W5, W6, W7 = drivers receive varying incentives week by week
  - Same drivers observed across all weeks

Model:
  completed_per_day_it = α_i + δ_t + β·Incentive_it + ε_it

  Where:
    α_i  = driver fixed effects  (absorbs each driver's baseline productivity)
    δ_t  = week fixed effects    (absorbs week-level demand shocks, weather, etc.)
    β    = causal effect of each incentive vs no-incentive weeks
    SE   = clustered by zone

  This is a two-way fixed effects (TWFE) DiD:
  "How many MORE orders per day did a driver complete in weeks they had
   incentive X compared to their own no-incentive baseline weeks?"

  OUTCOME: completed_per_day (orders / active days in that week)
  This normalises for W4 being 14 days vs ~7 days for other weeks.

Models:
  1. TWFE pooled          — main causal estimate (driver FE + week FE)
  2. Pooled OLS           — week FE only, no driver FE (comparison)
  3. TWFE per week        — heterogeneous effects over time
  4. TWFE × zone_type     — heterogeneous effects by win/lose/swing region

Outputs:
  did_incentive_results.csv      — tidy results (coef, SE, pval, CI per incentive)
  did_incentive_pivot.csv        — coef + pval side by side
  did_regression_summary.txt     — formatted regression table
  did_zone_heterogeneity.csv     — Model 4 interaction results by zone type
"""

import os
import re
import glob
import warnings
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

warnings.filterwarnings("ignore")

# ═══════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════

DATA_DIR   = "../mainexperiment/weekly/tier_merged_updated/"
OUTPUT_DIR = "./results"

# Use per-day outcome to normalise across weeks of different lengths
# The raw 'completed' column will be divided by active days per driver-week
OUTCOME_RAW   = "total_online_hour"           # raw column in data
OUTCOME       = "hours_per_day"   # derived outcome used in models

TREATMENT_COL = "Incentive"
DRIVER_ID_COL = "driver_id"
DAYS_COL      = "active_days"         # column with days active that week (if available)
                                       # if missing, will be inferred from row counts

# Zone type column — must contain values like 'Win', 'Lose', 'Swing'
# (exact strings — update to match your data)
ZONE_TYPE_COL = "status"

# The exact string in Incentive column meaning "no incentive"
REFERENCE_LEVEL = "No Incentive"

CLUSTER_BY = "status"

# Week labels from filenames — W1 must come first (baseline)
WEEK_MAP = [
    ("0710_0713", "W1_Jul10"),    # ← baseline: no incentive for everyone
    ("0714_0720", "W2_Jul14"),
    ("0721_0727", "W3_Jul21"),
    ("0728_0810", "W4_Jul28"),    # ← second reset: no incentive for everyone (14 days)
    ("0811_0815", "W5_Aug11"),
    ("0816_0822", "W6_Aug16"),
    ("0823_0829", "W7_Aug23"),
]

# Expected days per week (used to infer active_days if column is missing)
WEEK_DAYS = {
    "W1_Jul10": 4,
    "W2_Jul14": 7,
    "W3_Jul21": 7,
    "W4_Jul28": 14,   # two-week reset period
    "W5_Aug11": 5,
    "W6_Aug16": 7,
    "W7_Aug23": 7,
}

# ═══════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════

def safe(c):
    """Make column names safe for statsmodels formulas."""
    return re.sub(r"[^0-9a-zA-Z_]", "_", str(c))

def sig(p):
    if p < 0.01:  return "***"
    if p < 0.05:  return "**"
    if p < 0.10:  return "*"
    return ""

def load_files(data_dir):
    pattern = os.path.join(data_dir, "tiers_merged_all-*.csv")
    files   = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(
            f"No files matching 'tiers_merged_all-*.csv' found in: {os.path.abspath(data_dir)}"
        )
    dfs = []
    for fp in files:
        fname      = os.path.basename(fp)
        fname_norm = fname.replace("-", "_")
        week_label = next((lbl for pat, lbl in WEEK_MAP if pat in fname_norm), fname)
        df         = pd.read_csv(fp, low_memory=False)
        df["_week"] = week_label
        dfs.append(df)
        print(f"  {fname:<45} → {week_label}  ({len(df):,} rows)")
    out = pd.concat(dfs, ignore_index=True)
    print(f"\n  Total rows: {len(out):,}")
    return out

def aggregate_driver_week(df, outcome_raw_s, treatment_s, driver_s,
                          cluster_s, week_s, days_col_s, zone_type_s):
    """
    Collapse to one row per driver × week.
    - completed:       summed across days
    - active_days:     summed (or inferred from row count if column missing)
    - Incentive/zone:  mode (most common value that week)
    - zone_type:       mode
    """
    cat_cols = [c for c in [treatment_s, cluster_s, zone_type_s] if c and c in df.columns]
    has_days = days_col_s and days_col_s in df.columns

    agg_dict = {outcome_raw_s: "sum"}
    if has_days:
        agg_dict[days_col_s] = "sum"
    else:
        # proxy: count rows (each row = 1 driver-day)
        agg_dict["_row_count"] = ("_row_count", "sum") if "_row_count" in df.columns else None

    for c in cat_cols:
        agg_dict[c] = lambda x: x.mode().iloc[0] if len(x.mode()) else np.nan

    # Add a row count column to use as day proxy if needed
    df = df.copy()
    df["_row_count"] = 1

    if not has_days:
        agg_dict["_row_count"] = "sum"

    agg = df.groupby([driver_s, week_s], as_index=False).agg(agg_dict)

    # Build completed_per_day
    if has_days:
        # Use actual active days
        agg[OUTCOME] = agg[outcome_raw_s] / agg[days_col_s].replace(0, np.nan)
    else:
        # Use week-level expected days as denominator (more stable than row count)
        agg["_expected_days"] = agg[week_s].map(WEEK_DAYS).fillna(7)
        agg[OUTCOME] = agg[outcome_raw_s] / agg["_expected_days"]

    print(f"  Rows after aggregation: {len(agg):,}")
    print(f"  Unique drivers:         {agg[driver_s].nunique():,}")
    print(f"  Unique weeks:           {agg[week_s].nunique()}")
    return agg

def run_ols_clustered(formula, data, cluster_col):
    if cluster_col and cluster_col in data.columns and data[cluster_col].nunique() > 1:
        return smf.ols(formula, data=data).fit(
            cov_type="cluster",
            cov_kwds={"groups": data[cluster_col]},
        )
    return smf.ols(formula, data=data).fit(cov_type="HC3")

def demean_within_driver(panel, outcome_s, driver_s):
    """Within-transformation: demean by driver (equivalent to driver FE dummies)."""
    demeaned = (outcome_s + "_dm")
    panel[demeaned] = (
        panel[outcome_s]
        - panel.groupby(driver_s)[outcome_s].transform("mean")
        + panel[outcome_s].mean()
    )
    return panel, demeaned

def extract_incentive_coefs(model, treatment_s, ref):
    rows = []
    for param in model.params.index:
        if treatment_s in param and "Intercept" not in param:
            match = re.search(r"\[T\.(.*?)\]", param)
            label = match.group(1) if match else param
            ci    = model.conf_int().loc[param]
            rows.append({
                "incentive": label,
                "coef":    round(model.params[param],  4),
                "se":      round(model.bse[param],     4),
                "tstat":   round(model.tvalues[param], 4),
                "pval":    round(model.pvalues[param], 4),
                "ci_low":  round(ci[0], 4),
                "ci_high": round(ci[1], 4),
            })
    return rows

def print_coef_table(rows, ref):
    print(f"\n  {'Incentive':<38} {'Coef':>9} {'SE':>7} {'p':>7}  Sig  Interpretation")
    print("  " + "─" * 85)
    for r in sorted(rows, key=lambda x: -x["coef"]):
        direction = (f"+{r['coef']:.2f} orders/day vs {ref}"
                     if r["coef"] >= 0
                     else f"{r['coef']:.2f} orders/day vs {ref}")
        print(f"  {r['incentive']:<38} {r['coef']:>9.4f} {r['se']:>7.4f}"
              f" {r['pval']:>7.3f}  {sig(r['pval']):<3}  {direction}")

# ═══════════════════════════════════════════════════════
# MODEL 4 HELPERS
# ═══════════════════════════════════════════════════════

def extract_interaction_coefs(model, treatment_s, zone_type_s, ref):
    """
    Extract coefficients from the incentive × zone_type interaction model.

    Statsmodels encodes interactions like:
      C(Incentive)[T.Active Bonus]:C(zone_type)[T.Swing]

    This function parses those into tidy rows with columns:
      incentive, zone_type, coef, se, tstat, pval, ci_low, ci_high

    Main effects (incentive without zone interaction) are also extracted
    and labelled with zone_type = '<reference zone>' for clarity.
    """
    rows = []
    ci_df = model.conf_int()

    for param in model.params.index:
        # Skip intercept and pure week FE terms
        if "Intercept" in param or (week_s_global in param and treatment_s not in param
                                     and zone_type_s not in param):
            continue

        has_incentive  = treatment_s  in param
        has_zone       = zone_type_s  in param

        if not has_incentive:
            continue

        # Parse incentive label
        inc_match = re.search(rf"{treatment_s}\[T\.(.*?)\]", param)
        incentive_label = inc_match.group(1) if inc_match else param

        # Parse zone_type label (present only in interaction terms)
        zone_match = re.search(rf"{zone_type_s}\[T\.(.*?)\]", param)
        zone_label = zone_match.group(1) if zone_match else "reference_zone"

        ci = ci_df.loc[param]
        rows.append({
            "incentive":  incentive_label,
            "zone_type":  zone_label,
            "term":       "interaction" if has_zone else "main_effect",
            "coef":       round(model.params[param],  4),
            "se":         round(model.bse[param],     4),
            "tstat":      round(model.tvalues[param], 4),
            "pval":       round(model.pvalues[param], 4),
            "ci_low":     round(ci[0], 4),
            "ci_high":    round(ci[1], 4),
        })
    return rows

def compute_zone_marginal_effects(interaction_rows, zone_types, ref_zone):
    """
    Convert main effects + interactions into total marginal effects per zone.

    For the reference zone: effect = main_effect coef
    For other zones:        effect = main_effect coef + interaction coef

    This gives you the answer to "what is the effect of Active Bonus
    specifically in Win / Lose / Swing zones?"
    """
    df = pd.DataFrame(interaction_rows)
    main   = df[df["term"] == "main_effect"].set_index("incentive")
    inter  = df[df["term"] == "interaction"].set_index(["incentive", "zone_type"])

    results = []
    incentives = main.index.unique()

    for inc in incentives:
        # Reference zone — main effect only
        m = main.loc[inc]
        results.append({
            "incentive":     inc,
            "zone_type":     ref_zone,
            "total_coef":    m["coef"],
            "pval":          m["pval"],
            "note":          "main effect (reference zone)"
        })

        # Other zones — main effect + interaction
        for zone in zone_types:
            if zone == ref_zone:
                continue
            if (inc, zone) in inter.index:
                i = inter.loc[(inc, zone)]
                total = m["coef"] + i["coef"]
                results.append({
                    "incentive":     inc,
                    "zone_type":     zone,
                    "total_coef":    round(total, 4),
                    "interaction_coef": i["coef"],
                    "interaction_pval": i["pval"],
                    "pval":          i["pval"],   # significance of the *difference*
                    "note":          "main + interaction"
                })
            else:
                results.append({
                    "incentive":  inc,
                    "zone_type":  zone,
                    "total_coef": m["coef"],
                    "pval":       np.nan,
                    "note":       "no interaction term found"
                })

    return pd.DataFrame(results)

def print_zone_effects_table(marginal_df, ref_zone):
    print(f"\n  Total effect of each incentive by zone type")
    print(f"  (reference zone = '{ref_zone}'; significance = interaction p-value for non-ref zones)")
    print(f"\n  {'Incentive':<28} {'Zone':<10} {'Effect':>8}  Sig  Note")
    print("  " + "─" * 70)
    for _, r in marginal_df.sort_values(["incentive", "zone_type"]).iterrows():
        p = r.get("pval", np.nan)
        s = sig(p) if pd.notna(p) else ""
        print(f"  {r['incentive']:<28} {r['zone_type']:<10} {r['total_coef']:>8.3f}  {s:<3}  {r['note']}")

# ═══════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════

# Global reference for week column name (used in interaction extractor)
week_s_global = "_week"

def main():
    global week_s_global

    print("=" * 70)
    print("Thailand Incentive Experiment — DiD Panel Regression")
    print("Two-Way Fixed Effects: Driver FE + Week FE")
    print("Outcome: completed orders per active day")
    print("=" * 70 + "\n")

    # ── Load ──────────────────────────────────────────────
    df = load_files(DATA_DIR)

    # ── Safe column names ─────────────────────────────────
    needed = [OUTCOME_RAW, TREATMENT_COL, DRIVER_ID_COL, CLUSTER_BY,
              DAYS_COL, ZONE_TYPE_COL, "_week"]
    rename = {c: safe(c) for c in needed if c and c in df.columns}
    df = df.rename(columns=rename)

    outcome_raw_s = rename.get(OUTCOME_RAW,   safe(OUTCOME_RAW))
    outcome_s     = safe(OUTCOME)
    treatment_s   = rename.get(TREATMENT_COL, safe(TREATMENT_COL))
    driver_s      = rename.get(DRIVER_ID_COL, safe(DRIVER_ID_COL))
    cluster_s     = rename.get(CLUSTER_BY,    safe(CLUSTER_BY)) if CLUSTER_BY else None
    days_col_s    = rename.get(DAYS_COL,      safe(DAYS_COL))   if DAYS_COL else None
    zone_type_s   = rename.get(ZONE_TYPE_COL, safe(ZONE_TYPE_COL)) if ZONE_TYPE_COL else None
    week_s        = "_week"
    week_s_global = week_s

    df[outcome_raw_s] = pd.to_numeric(df[outcome_raw_s], errors="coerce")

    # ── Incentive distribution ────────────────────────────
    print("\n" + "─" * 70)
    print("Incentive × Week distribution (driver-day rows):")
    cross = pd.crosstab(df[week_s], df[treatment_s])
    print(cross.to_string())

    # ── Zone type distribution ────────────────────────────
    if zone_type_s and zone_type_s in df.columns:
        print("\nZone type distribution:")
        print(df[zone_type_s].value_counts().to_string())
    else:
        print(f"\n  ⚠ Column '{ZONE_TYPE_COL}' not found — Model 4 will be skipped.")
        zone_type_s = None

    # ── Resolve reference ─────────────────────────────────
    ref = REFERENCE_LEVEL
    if ref not in df[treatment_s].values:
        w1 = next((lbl for _, lbl in WEEK_MAP if lbl in df[week_s].unique()), None)
        ref = df[df[week_s] == w1][treatment_s].mode().iloc[0] if w1 else \
              df[treatment_s].value_counts().idxmax()
        print(f"\n  '{REFERENCE_LEVEL}' not found — using '{ref}' as reference")
    print(f"\n  ✓ Reference (baseline): '{ref}'")

    # ── Aggregate to driver × week ────────────────────────
    print("\n" + "─" * 70)
    print("Aggregating to driver × week panel...")
    panel = aggregate_driver_week(df, outcome_raw_s, treatment_s, driver_s,
                                  cluster_s, week_s, days_col_s, zone_type_s)
    panel = panel.dropna(subset=[outcome_s, treatment_s])

    # ── Sanity checks ─────────────────────────────────────
    print("\n" + "─" * 70)
    print("Panel sanity checks:")
    obs_per_driver = panel.groupby(driver_s).size()
    print(f"  Weeks observed per driver — mean: {obs_per_driver.mean():.1f} "
          f"min: {obs_per_driver.min()}  max: {obs_per_driver.max()}")

    print("\n  Incentive × Week counts (driver-week rows):")
    cross2 = pd.crosstab(panel[week_s], panel[treatment_s])
    print(cross2.to_string())

    print("\n  Mean completed orders/day by incentive (raw, unadjusted):")
    means = panel.groupby(treatment_s)[outcome_s].agg(["mean", "count"]).round(3)
    means.columns = ["mean_completed_per_day", "n_driver_weeks"]
    print(means.to_string())

    if zone_type_s and zone_type_s in panel.columns:
        print("\n  No-incentive driver zone composition by week:")
        no_inc = panel[panel[treatment_s] == ref]
        zone_cross = pd.crosstab(no_inc[week_s], no_inc[zone_type_s])
        print(zone_cross.to_string())
        print("  ↑ Check if W5/W6 no-incentive drivers are concentrated in Win zones")

    # ── Driver FE demeaning ───────────────────────────────
    panel, demeaned_col = demean_within_driver(panel, outcome_s, driver_s)

    treatment_expr = f"C({treatment_s}, Treatment('{ref}'))"

    # ══════════════════════════════════════════════════════
    # MODEL 1 — TWFE DiD (driver FE + week FE)
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("MODEL 1: Two-Way Fixed Effects DiD")
    print(f"  {OUTCOME} ~ C(Incentive) + C(driver_id) + C(week)")
    print("=" * 70)

    formula_twfe  = f"{demeaned_col} ~ {treatment_expr} + C({week_s})"
    twfe_model    = run_ols_clustered(formula_twfe, panel, cluster_s)
    twfe_rows     = extract_incentive_coefs(twfe_model, treatment_s, ref)

    print(f"\n  n={int(twfe_model.nobs):,}  drivers={panel[driver_s].nunique():,}  "
          f"R²={twfe_model.rsquared:.3f}  Adj-R²={twfe_model.rsquared_adj:.3f}")
    print_coef_table(twfe_rows, ref)

    # ══════════════════════════════════════════════════════
    # MODEL 2 — Pooled OLS (week FE only)
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("MODEL 2: Pooled OLS (week FE only, no driver FE) — for comparison")
    print("=" * 70)

    formula_pooled = f"{outcome_s} ~ {treatment_expr} + C({week_s})"
    pooled_model   = run_ols_clustered(formula_pooled, panel, cluster_s)
    pooled_rows    = extract_incentive_coefs(pooled_model, treatment_s, ref)

    print(f"\n  n={int(pooled_model.nobs):,}  "
          f"R²={pooled_model.rsquared:.3f}  Adj-R²={pooled_model.rsquared_adj:.3f}")
    print_coef_table(pooled_rows, ref)

    print("\n  ↑ If MODEL 1 vs MODEL 2 coefficients shift a lot,")
    print("    driver FE is doing important work (selection bias exists).")

    # ══════════════════════════════════════════════════════
    # MODEL 3 — TWFE per week (heterogeneous effects over time)
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("MODEL 3: TWFE per week (heterogeneous treatment effects over time)")
    print("=" * 70)

    week_order    = [lbl for _, lbl in WEEK_MAP if lbl in panel[week_s].unique()]
    week_models   = {}
    all_week_rows = []

    for week in week_order:
        sub = panel[
            (panel[week_s] == week) |
            (panel[treatment_s] == ref)
        ].copy()

        sub, sub_dm = demean_within_driver(sub, outcome_s, driver_s)

        n_incentives = sub[sub[week_s] == week][treatment_s].nunique()
        if n_incentives < 2:
            only = sub[sub[week_s] == week][treatment_s].iloc[0]
            label = "skipped (all on No Incentive)" if only == ref else \
                    f"skipped (only incentive: '{only}')"
            print(f"\n  {week}: {label}")
            continue

        try:
            formula_w = f"{sub_dm} ~ {treatment_expr} + C({week_s})"
            model_w   = run_ols_clustered(formula_w, sub, cluster_s)
            week_models[week] = model_w

            rows = extract_incentive_coefs(model_w, treatment_s, ref)
            week_incentives = sub[sub[week_s] == week][treatment_s].unique()
            rows = [r for r in rows if r["incentive"] in week_incentives]

            for r in rows:
                r.update({"week": week, "n_obs": int(model_w.nobs),
                           "r2": round(model_w.rsquared, 4), "reference": ref,
                           "model": "TWFE_by_week"})
            all_week_rows.extend(rows)

            print(f"\n  {week}  (n={int(model_w.nobs):,}  R²={model_w.rsquared:.3f})")
            print_coef_table(rows, ref)

        except Exception as e:
            print(f"\n  {week}: ERROR — {e}")

    # ══════════════════════════════════════════════════════
    # MODEL 4 — TWFE × zone_type (heterogeneous effects by region)
    #
    # Formula:
    #   completed_per_day_dm ~ C(Incentive) * C(zone_type) + C(week)
    #
    # This adds an interaction between incentive and zone type on top of
    # the driver-demeaned outcome (driver FE already absorbed via demeaning).
    #
    # Interpretation:
    #   Main effect of incentive    = effect in the REFERENCE zone
    #   Interaction term            = HOW MUCH MORE/LESS the effect is in
    #                                 non-reference zones vs the reference zone
    #   Total effect in a zone      = main effect + interaction
    #                                 (computed in compute_zone_marginal_effects)
    #
    # This answers: "Does Active Bonus work better in Swing zones than Win zones?"
    # ══════════════════════════════════════════════════════
    zone_rows_all   = []
    marginal_df_out = None

    if zone_type_s and zone_type_s in panel.columns:
        print("\n" + "=" * 70)
        print("MODEL 4: TWFE × Zone Type (heterogeneous effects by Win/Lose/Swing)")
        print(f"  {OUTCOME}_dm ~ C(Incentive) * C(zone_type) + C(week)")
        print("  Main effect   = incentive effect in the REFERENCE zone")
        print("  Interaction   = differential effect in other zones vs reference")
        print("  Total effect  = main + interaction (reported in marginal table)")
        print("=" * 70)

        zone_types = sorted(panel[zone_type_s].dropna().unique())

        # Choose reference zone — use the largest group for stability
        ref_zone = panel[zone_type_s].value_counts().idxmax()
        zone_type_expr = f"C({zone_type_s}, Treatment('{ref_zone}'))"

        print(f"\n  Zone types found: {zone_types}")
        print(f"  Reference zone (largest): '{ref_zone}'")
        print(f"\n  Zone × Week distribution:")
        if zone_type_s in panel.columns:
            print(pd.crosstab(panel[week_s], panel[zone_type_s]).to_string())

        formula_zone = (
            f"{demeaned_col} ~ "
            f"{treatment_expr} * {zone_type_expr} + "
            f"C({week_s})"
        )

        try:
            zone_model = run_ols_clustered(formula_zone, panel, cluster_s)

            print(f"\n  n={int(zone_model.nobs):,}  "
                  f"R²={zone_model.rsquared:.3f}  "
                  f"Adj-R²={zone_model.rsquared_adj:.3f}")

            # ── Raw interaction coefficients ───────────────
            zone_rows = extract_interaction_coefs(zone_model, treatment_s,
                                                   zone_type_s, ref)
            zone_rows_all = zone_rows

            print(f"\n  Raw coefficients (main effects + interactions):")
            print(f"  {'Term':<60} {'Coef':>9} {'SE':>7} {'p':>7}  Sig")
            print("  " + "─" * 90)
            for r in zone_rows:
                term_label = f"{r['incentive']} × {r['zone_type']}" \
                             if r["term"] == "interaction" \
                             else f"{r['incentive']} [main / ref zone]"
                print(f"  {term_label:<60} {r['coef']:>9.4f} {r['se']:>7.4f}"
                      f" {r['pval']:>7.3f}  {sig(r['pval']):<3}")

            # ── Marginal total effects per zone ───────────
            print(f"\n  ── Marginal total effects (main + interaction) ──")
            marginal_df = compute_zone_marginal_effects(zone_rows, zone_types, ref_zone)
            marginal_df_out = marginal_df
            print_zone_effects_table(marginal_df, ref_zone)

            # ── Pivot: incentive × zone for easy scanning ──
            print(f"\n  ── Pivot: total effect by incentive × zone ──")
            pivot = marginal_df.pivot_table(
                index="incentive", columns="zone_type", values="total_coef"
            )
            print(pivot.round(3).to_string())

            print(f"\n  ── Interpretation guide ──")
            print(f"  • Positive value = incentive RAISES orders/day vs No Incentive")
            print(f"  • Interaction p-value = is the zone difference statistically significant?")
            print(f"  • Large variation across columns = zone type matters a lot")
            print(f"  • Small variation across columns = incentive effect is uniform")

        except Exception as e:
            print(f"\n  Model 4 ERROR — {e}")
            import traceback
            traceback.print_exc()

    else:
        print("\n  MODEL 4 skipped — zone_type column not available.")
        print(f"  Add '{ZONE_TYPE_COL}' column to your data to enable this analysis.")

    # ══════════════════════════════════════════════════════
    # SAVE OUTPUTS
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("Saving outputs...")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Tag model rows
    for r in twfe_rows:
        r.update({"week": "TWFE_POOLED", "n_obs": int(twfe_model.nobs),
                   "r2": round(twfe_model.rsquared, 4), "reference": ref,
                   "model": "TWFE"})
    for r in pooled_rows:
        r.update({"week": "OLS_POOLED", "n_obs": int(pooled_model.nobs),
                   "r2": round(pooled_model.rsquared, 4), "reference": ref,
                   "model": "OLS_no_driverFE"})

    results_df = pd.DataFrame(twfe_rows + pooled_rows + all_week_rows)
    results_df = results_df[[
        "model", "week", "incentive", "coef", "se", "tstat",
        "pval", "ci_low", "ci_high", "n_obs", "r2", "reference"
    ]].round(4)

    csv1 = os.path.join(OUTPUT_DIR, "did_incentive_results_hours_worked.csv")
    results_df.to_csv(csv1, index=False)
    print(f"\n  ✓ {csv1}")

    # Pivot: TWFE weekly coefs × incentive
    week_rows_df = pd.DataFrame(all_week_rows)
    if not week_rows_df.empty:
        pivot_coef = week_rows_df.pivot_table(index="incentive", columns="week", values="coef")
        pivot_pval = week_rows_df.pivot_table(index="incentive", columns="week", values="pval")

        actual_weeks = list(pivot_coef.columns)
        ordered_cols = [w for w in week_order if w in actual_weeks]
        pivot_coef = pivot_coef[ordered_cols]
        pivot_pval = pivot_pval[ordered_cols]

        twfe_series = pd.Series({r["incentive"]: r["coef"] for r in twfe_rows})
        pval_series = pd.Series({r["incentive"]: r["pval"] for r in twfe_rows})
        pivot_coef.insert(0, "TWFE_POOLED", twfe_series)
        pivot_pval.insert(0, "TWFE_POOLED", pval_series)

        csv2 = os.path.join(OUTPUT_DIR, "did_incentive_pivot_hours_worked.csv")
        with open(csv2, "w") as f:
            f.write(f"COEFFICIENTS (orders/day) — vs '{ref}' (driver FE + week FE)\n")
            pivot_coef.round(4).to_csv(f)
            f.write(f"\nP-VALUES\n")
            pivot_pval.round(4).to_csv(f)
        print(f"  ✓ {csv2}")

    # Zone heterogeneity CSV
    if zone_rows_all:
        csv3 = os.path.join(OUTPUT_DIR, "did_zone_heterogeneity_hours_worked.csv")
        zone_raw_df = pd.DataFrame(zone_rows_all)
        zone_raw_df["model"]     = "TWFE_zone_interaction"
        zone_raw_df["reference"] = ref
        zone_raw_df.round(4).to_csv(csv3, index=False)
        print(f"  ✓ {csv3}  (raw interaction coefficients)")

    if marginal_df_out is not None:
        csv4 = os.path.join(OUTPUT_DIR, "did_zone_marginal_effects_hours_worked.csv")
        marginal_df_out.round(4).to_csv(csv4, index=False)
        print(f"  ✓ {csv4}  (total effects per incentive × zone)")

    # Regression summary txt
    try:
        all_models      = [twfe_model, pooled_model] + list(week_models.values())
        all_model_names = ["TWFE", "OLS"] + list(week_models.keys())
        incent_params   = [p for p in twfe_model.params.index if treatment_s in p]

        summary = summary_col(
            all_models,
            model_names=all_model_names,
            stars=True,
            float_format="%.3f",
            regressor_order=incent_params,
            info_dict={
                "N":  lambda m: f"{int(m.nobs):,}",
                "R²": lambda m: f"{m.rsquared:.3f}",
            },
        )
        txt = os.path.join(OUTPUT_DIR, "did_regression_summary_hours_worked.txt")
        with open(txt, "w") as f:
            f.write("Two-Way Fixed Effects DiD — Thailand Incentive Experiment\n")
            f.write(f"Outcome: completed orders per active day\n")
            f.write(f"Reference: '{ref}' | SE clustered by {CLUSTER_BY}\n")
            f.write("TWFE = driver FE (within-demeaned) + week FE\n")
            f.write("OLS  = week FE only (no driver FE) — comparison\n\n")
            f.write(str(summary))
        print(f"  ✓ {txt}")
    except Exception as e:
        print(f"  (summary table skipped: {e})")

    print("\n" + "=" * 70)
    print("Done ✓")
    print("─" * 70)
    print("Outputs:")
    print("  did_incentive_results.csv      — full tidy results (Models 1–3)")
    print("  did_incentive_pivot.csv        — incentive × week coefficient matrix")
    print("  did_regression_summary.txt     — paper-style table")
    print("  did_zone_heterogeneity.csv     — Model 4 raw interaction coefficients")
    print("  did_zone_marginal_effects.csv  — Model 4 total effects per zone")
    print("─" * 70)
    print("How to read results:")
    print(f"  coef = extra completed orders/day vs '{ref}',")
    print("         controlling for each driver's own baseline (TWFE model)")
    print("  Model 4: interaction p-value = is the zone difference significant?")
    print("  *** p<0.01  ** p<0.05  * p<0.10")
    print("=" * 70)


if __name__ == "__main__":
    main()

Thailand Incentive Experiment — DiD Panel Regression
Two-Way Fixed Effects: Driver FE + Week FE
Outcome: completed orders per active day

  tiers_merged_all-0710-0713.csv                → W1_Jul10  (15,566 rows)
  tiers_merged_all-0714-0720.csv                → W2_Jul14  (29,301 rows)
  tiers_merged_all-0721-0727.csv                → W3_Jul21  (29,583 rows)
  tiers_merged_all-0728-0810.csv                → W4_Jul28  (63,437 rows)
  tiers_merged_all-0811-0815.csv                → W5_Aug11  (20,559 rows)
  tiers_merged_all-0816-0822.csv                → W6_Aug16  (30,336 rows)
  tiers_merged_all-0823-0829.csv                → W7_Aug23  (30,149 rows)

  Total rows: 218,931

──────────────────────────────────────────────────────────────────────
Incentive × Week distribution (driver-day rows):
Incentive  Active Bonus  Daily Productivity  Guarantee  No Incentive  Streak
_week                                                                       
W1_Jul10              0                   0   

In [14]:
pd.read_csv('./results/did_incentive_results_hours_worked.csv')

,model,week,incentive,coef,se,tstat,pval,ci_low,ci_high,n_obs,r2,reference
0,TWFE,TWFE_POOLED,Active Bonus,0.6906,0.1488,4.6418,0.0000,0.3990,0.9822,41231,0.0712,No Incentive
1,TWFE,TWFE_POOLED,Daily Productivity,0.5709,0.1083,5.2706,0.0000,0.3586,0.7832,41231,0.0712,No Incentive
2,TWFE,TWFE_POOLED,Guarantee,1.2428,0.1844,6.7381,0.0000,0.8813,1.6042,41231,0.0712,No Incentive
3,TWFE,TWFE_POOLED,Streak,0.7320,0.1569,4.6660,0.0000,0.4245,1.0395,41231,0.0712,No Incentive
4,OLS_no_driverFE,OLS_POOLED,Active Bonus,0.8073,0.2048,3.9424,0.0001,0.4059,1.2086,41231,0.0314,No Incentive
5,OLS_no_driverFE,OLS_POOLED,Daily Productivity,0.7836,0.1465,5.3489,0.0000,0.4965,1.0707,41231,0.0314,No Incentive
6,OLS_no_driverFE,OLS_POOLED,Guarantee,1.5088,0.0873,17.2798,0.0000,1.3377,1.6800,41231,0.0314,No Incentive
7,OLS_no_driverFE,OLS_POOLED,Streak,0.7350,0.1866,3.9385,0.0001,0.3692,1.1008,41231,0.0314,No Incentive
8,TWFE_by_week,W2_Jul14,Active Bonus,0.6706,0.0675,9.9355,0.0000,0.5383,0.8029,22830,0.0578,No Incentive
9,TWFE_by_week,W2_Jul14,Daily Productivity,0.6045,0.0338,17.8597,0.0000,0.5382,0.6709,22830,0.0578,No Incentive
